# 4. Prepare motion windows and spatial features

For each AVI in `data/raw/videos/`, run VideoMeta motion gating and object tracking. It writes a `motion_scores.png` graph, annotated window MP4s, spatial feature files, and `prepared_windows.json` to `data/interim/<video-name>/`.

YOLO tracking can be limited to selected class IDs via `DetectionConfig.classes`. Set that to `None` or `[]` to use every class known to the loaded model.

Import `videometa` from [TestPyPI](https://test.pypi.org/project/videometa/) only (`videometa>=0.0.9`). Do not install the local `videometa` source tree. After that TestPyPI release is published, refresh the environment with `uv sync`. Run this notebook with the VideoMeta vision dependencies installed.

In [4]:
import json
from pathlib import Path
from statistics import mean, median

import matplotlib.pyplot as plt
from videometa import (
    DetectionConfig,
    MotionGateConfig,
    ObjectBoundaryExtractor,
    RelevantWindowFinder,
    WindowSpatialFeatureJoiner,
    resolve_video_source,
)

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
RAW_VIDEOS = PROJECT_ROOT / "data" / "raw" / "videos"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MAX_VIDEOS = None  # Set an integer for a smaller experiment.
MAX_WINDOWS = None  # Set an integer to keep only the highest-motion windows.

motion_config = MotionGateConfig(
    window_seconds=12,
    stride_seconds=10,
    motion_threshold="std",  # or a float / "avg" / "median"
    motion_std_k=0.5,
    motion_std_direction="upper",  # "upper", "lower", or "both"
    warmup_seconds=2,
    mog_history=100,
)
window_finder = RelevantWindowFinder(motion_config)

# MEVA-compatible COCO class IDs. Set to None or [] to track every YOLO class.
meva_complete_classes = [
    # Actors & Vehicles
    0,   # person
    1,   # bicycle
    2,   # car
    3,   # motorcycle
    5,   # bus
    7,   # truck
    # Infrastructure & Animals
    9,   # traffic light
    10,  # fire hydrant
    11,  # stop sign
    12,  # parking meter
    13,  # bench
    16,  # dog
    # Carried Props & Bags
    24,  # backpack
    25,  # umbrella
    26,  # handbag
    28,  # suitcase
    39,  # bottle
    41,  # cup
    73,  # book
    # Tools & Sports Items
    32,  # sports ball
    34,  # baseball bat
    36,  # skateboard
    43,  # knife
    # Furniture & Screens
    56,  # chair
    57,  # couch
    60,  # dining table (desks / work surfaces)
    62,  # tv (monitors / screens)
    # Electronics
    63,  # laptop
    64,  # mouse
    66,  # keyboard
    67,  # cell phone
]
boundary_extractor = ObjectBoundaryExtractor(
    DetectionConfig(
        model_path="yolo26n.pt",
        confidence_threshold=0.25,
        spatial_grid=3,
        classes=meva_complete_classes,
    )
)

video_paths = sorted(RAW_VIDEOS.glob("*.avi"))[:MAX_VIDEOS]
if not video_paths:
    raise FileNotFoundError(f"No AVI files found in {RAW_VIDEOS}")

for video_path in video_paths:
    video_name = video_path.stem.removesuffix(".r13")
    output_dir = INTERIM_DIR / video_name
    frames_dir = output_dir / "window_frames"
    output_dir.mkdir(parents=True, exist_ok=True)

    source_path = resolve_video_source(video_path)
    video_info, motion_samples = window_finder.sample_motion(source_path)
    windows = window_finder.build_windows(motion_samples, video_info.duration_seconds)
    selected_windows = sorted(
        (window for window in windows if window.is_relevant),
        key=lambda window: window.peak_motion,
        reverse=True,
    )[:MAX_WINDOWS]
    scores = [sample.score for sample in motion_samples]
    average_score = mean(scores)
    median_score = median(scores)
    lower_threshold, upper_threshold = window_finder.resolve_motion_thresholds(motion_samples)

    timestamps = [sample.timestamp_seconds for sample in motion_samples]
    matching_samples = [
        sample
        for sample in motion_samples
        if (lower_threshold is not None and sample.score < lower_threshold)
        or (upper_threshold is not None and sample.score > upper_threshold)
    ]

    figure, axis = plt.subplots(figsize=(14, 4))
    axis.plot(timestamps, scores, linewidth=0.75, label="motion score")
    axis.scatter(
        [sample.timestamp_seconds for sample in matching_samples],
        [sample.score for sample in matching_samples],
        color="darkorange",
        s=14,
        zorder=3,
        label="scores in selected range",
    )
    axis.axhline(average_score, color="darkviolet", linestyle=":", label="average")
    axis.axhline(median_score, color="teal", linestyle=":", label="median")
    if lower_threshold is not None:
        axis.axhline(lower_threshold, color="royalblue", linestyle="--", label="lower threshold")
    if upper_threshold is not None:
        axis.axhline(upper_threshold, color="crimson", linestyle="--", label="upper threshold")
    for window in selected_windows:
        axis.axvspan(window.start_seconds, window.end_seconds, color="seagreen", alpha=0.15)
    axis.set(title=f"Motion scores: {video_name}", xlabel="Time (seconds)", ylabel="Motion score")
    axis.legend()
    figure.savefig(output_dir / "motion_scores.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

    figure, (box_axis, distribution_axis) = plt.subplots(1, 2, figsize=(14, 4))
    box_axis.boxplot(scores, vert=True, showfliers=True)
    box_axis.axhline(average_score, color="darkviolet", linestyle=":", label="average")
    box_axis.axhline(median_score, color="teal", linestyle=":", label="median")
    if lower_threshold is not None:
        box_axis.axhline(lower_threshold, color="royalblue", linestyle="--", label="lower threshold")
    if upper_threshold is not None:
        box_axis.axhline(upper_threshold, color="crimson", linestyle="--", label="upper threshold")
    box_axis.set(title="Motion-score box plot", ylabel="Motion score", xticks=[])
    box_axis.legend()

    distribution_axis.hist(scores, bins="auto", color="steelblue", edgecolor="white")
    distribution_axis.axvline(average_score, color="darkviolet", linestyle=":", label="average")
    distribution_axis.axvline(median_score, color="teal", linestyle=":", label="median")
    if lower_threshold is not None:
        distribution_axis.axvline(lower_threshold, color="royalblue", linestyle="--", label="lower threshold")
    if upper_threshold is not None:
        distribution_axis.axvline(upper_threshold, color="crimson", linestyle="--", label="upper threshold")
    distribution_axis.set(title="Motion-score distribution", xlabel="Motion score", ylabel="Frame count")
    distribution_axis.legend()
    figure.suptitle(f"Motion-score summary: {video_name}")
    figure.tight_layout()
    figure.savefig(output_dir / "motion_score_distribution.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

    object_annotations = boundary_extractor.extract(source_path, selected_windows, video_info.fps)

    joiner = WindowSpatialFeatureJoiner(
        output_directory=frames_dir,
        annotated_video_size=(640, 360),
        annotated_video_fps=4,
        annotated_video_max_frames=32,
        encode_frame_images=False,
    )
    prepared_windows = [joiner.prepare(source_path, annotation) for annotation in object_annotations]
    manifest = [
        {
            "start_seconds": item.start_seconds,
            "end_seconds": item.end_seconds,
            "object_features": item.object_features,
            "frame_features": item.frame_features,
            "image_messages": item.image_messages,
            "artifact_directory": item.artifact_directory,
            "annotated_video_path": item.annotated_video_path,
        }
        for item in prepared_windows
    ]
    (output_dir / "prepared_windows.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Prepared {len(prepared_windows)} windows for {video_name}")

/var/folders/0c/sn1ftfcs24l8fqqvjpwlpt1w0000gn/T/ipykernel_34162/3289610769.py:94: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box_axis.boxplot(scores, vert=True, showfliers=True)


Prepared 15 windows for 2018-03-05.13-15-00.13-20-00.bus.G340


/var/folders/0c/sn1ftfcs24l8fqqvjpwlpt1w0000gn/T/ipykernel_34162/3289610769.py:94: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box_axis.boxplot(scores, vert=True, showfliers=True)


Prepared 15 windows for 2018-03-05.13-15-00.13-20-00.bus.G505


/var/folders/0c/sn1ftfcs24l8fqqvjpwlpt1w0000gn/T/ipykernel_34162/3289610769.py:94: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box_axis.boxplot(scores, vert=True, showfliers=True)


...

Prepared 9 windows for 2018-03-05.13-20-00.13-25-00.bus.G340


/var/folders/0c/sn1ftfcs24l8fqqvjpwlpt1w0000gn/T/ipykernel_34162/3289610769.py:94: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box_axis.boxplot(scores, vert=True, showfliers=True)


Prepared 10 windows for 2018-03-05.13-20-00.13-25-00.school.G336


/var/folders/0c/sn1ftfcs24l8fqqvjpwlpt1w0000gn/T/ipykernel_34162/3289610769.py:94: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box_axis.boxplot(scores, vert=True, showfliers=True)


Prepared 30 windows for 2018-03-05.13-20-00.13-25-00.school.G423
